# RoBERTa-based Manipulation Detection (Binary Classification) with LoRA and Freeze layers integrated
This notebook uses `roberta-base` to classify dialogue as manipulative or not using the MentalManip dataset.

In [1]:
!pip install -q transformers
!pip install -q datasets
!pip install -q evaluate
## transformers upgrade
!pip install -q --upgrade transformers

## Datasets need upgrading to work
!pip install -q --upgrade datasets


In [2]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import evaluate



In [3]:
# Load the MentalManip dataset (binary classification)
# Load dataset
dataset = load_dataset("audreyeleven/MentalManip", name="mentalmanip_maj")

print(dataset)


Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'manipulative', 'technique', 'vulnerability'],
        num_rows: 4000
    })
})


In [4]:
# Ensure the 'manipulative' column is class-labeled
dataset = dataset.class_encode_column("manipulative")




In [5]:
model_ckpt = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(example):
    return tokenizer(example["dialogue"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
print(tokenized)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'manipulative', 'technique', 'vulnerability', 'input_ids', 'attention_mask'],
        num_rows: 4000
    })
})


In [6]:
# Split the dataset into training and testing sets
train_test_split = tokenized["train"].train_test_split(test_size=0.2) # Adjust the test_size as needed

# Update the tokenized dataset with the new splits
tokenized["train"] = train_test_split["train"]
tokenized["test"] = train_test_split["test"]

# Make the data work with the nomenclature
tokenized = tokenized.rename_column("manipulative", "labels")

print(tokenized)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'labels', 'technique', 'vulnerability', 'input_ids', 'attention_mask'],
        num_rows: 3200
    })
    test: Dataset({
        features: ['id', 'dialogue', 'labels', 'technique', 'vulnerability', 'input_ids', 'attention_mask'],
        num_rows: 800
    })
})


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=2  # Binary classification
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:

# Install PEFT if not already installed
!pip install -q peft


In [9]:

# Import LoRA-specific modules
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)

# Freeze all layers
for param in base_model.parameters():
    param.requires_grad = False

# Unfreeze last two encoder layers, and classifier
for i in [-1, -2]:  # last two encoder blocks
    for param in base_model.roberta.encoder.layer[i].parameters():
        param.requires_grad = True

for param in base_model.classifier.parameters():
    param.requires_grad = True


# Define LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters to confirm LoRA setup
model.print_trainable_parameters()


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [10]:
## Training args

training_args = TrainingArguments(
    output_dir="./bert-binary-manip",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    run_name="bert-binary-manip",
    report_to="none",
)


In [11]:
## Evaluation metrics

import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall = evaluate.load('recall')
f1 = evaluate.load('f1')

def compute_metrics(p):
    predictions, labels = p
    predictions_argmax = np.argmax(predictions, axis=1)

    return {
        "accuracy": accuracy.compute(predictions=predictions_argmax, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions_argmax, references=labels, average='weighted')["precision"],
        "recall": recall.compute(predictions=predictions_argmax, references=labels, average='weighted')["recall"],
        "f1": f1.compute(predictions=predictions_argmax, references=labels, average='weighted')["f1"],
    }

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()



/tmp/ipython-input-499019960.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.609300,0.585380,0.720000,0.518400,0.720000,0.602791
2,0.532900,0.544806,0.736250,0.719334,0.736250,0.661734
3,0.659700,0.547119,0.725000,0.700903,0.725000,0.706093
4,0.580000,0.544336,0.727500,0.694021,0.727500,0.692359


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.

TrainOutput(global_step=800, training_loss=0.5760229581594467, metrics={'train_runtime': 234.7188, 'train_samples_per_second': 54.533, 'train_steps_per_second': 3.408, 'total_flos': 850675354828800.0, 'train_loss': 0.5760229581594467, 'epoch': 4.0})

In [13]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
# Change current working directory to a project folder in Drive
#################################
### Change path to match your repo
###################################
%cd "/content/drive/My Drive/Colab Notebooks/266/FinalProject/SpamAssasin"

####################################

Mounted at /content/drive
/content/drive/My Drive/Colab Notebooks/266/FinalProject/SpamAssasin


In [14]:

# Save model and tokenizer
model_path = "./saved_roberta_lora_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)


('./saved_roberta_lora_model/tokenizer_config.json',
 './saved_roberta_lora_model/special_tokens_map.json',
 './saved_roberta_lora_model/vocab.json',
 './saved_roberta_lora_model/merges.txt',
 './saved_roberta_lora_model/added_tokens.json',
 './saved_roberta_lora_model/tokenizer.json')

In [15]:

##  Easy eval results...
##  Get the log history from the trainer state
log_history = trainer.state.log_history

# Initialize placeholders
last_train_loss = None
last_eval_loss = None

# Iterate through log history to find the last recorded train and eval loss
for log in reversed(log_history):
    if last_eval_loss is None and "eval_loss" in log:
        last_eval_loss = log["eval_loss"]
    if last_train_loss is None and "loss" in log:
        last_train_loss = log["loss"]
    if last_train_loss is not None and last_eval_loss is not None:
        break

# Calculate overfitting ratio
if last_train_loss is not None and last_eval_loss is not None:
    ratio = last_train_loss / last_eval_loss
    print(f"Training Loss: {last_train_loss:.5f}")
    print(f"Validation Loss: {last_eval_loss:.5f}")
    print(f"Overfitting Ratio: {ratio:.5f}")
    if ratio < 0.6:
        print("Overfitting detected!")
    else:
        print("No significant overfitting.")
else:
    print("Could not find both training and evaluation loss in log history.")


Training Loss: 0.58000
Validation Loss: 0.54434
Overfitting Ratio: 1.06552
No significant overfitting.


In [16]:
# Predict on test set
preds = trainer.predict(tokenized["test"])
y_pred = preds.predictions.argmax(-1)
y_true = preds.label_ids

# Detailed classification report
print(classification_report(y_true, y_pred, target_names=["non-manipulative", "manipulative"]))


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


                  precision    recall  f1-score   support

non-manipulative       0.53      0.25      0.34       224
    manipulative       0.76      0.91      0.83       576

        accuracy                           0.73       800
       macro avg       0.64      0.58      0.59       800
    weighted avg       0.69      0.73      0.69       800



In [17]:
import pandas as pd
import numpy as np

# Get predictions
predictions = trainer.predict(tokenized["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

# Decode input_ids from the tokenized test set
decoded_texts = tokenizer.batch_decode(tokenized["test"]["input_ids"], skip_special_tokens=True)

# Create DataFrame with predictions and text
results_df = pd.DataFrame({
    "text": decoded_texts,
    "true_label": y_true,
    "predicted_label": y_pred
})

# Identify true positives, false negatives, false positives, and true negatives
TP_df = results_df[(results_df.true_label == 1) & (results_df.predicted_label == 1)]
FN_df = results_df[(results_df.true_label == 1) & (results_df.predicted_label == 0)]
FP_df = results_df[(results_df.true_label == 0) & (results_df.predicted_label == 1)]
TN_df = results_df[(results_df.true_label == 0) & (results_df.predicted_label == 0)]

# Sample up to 2 examples from each category, if available
TP = TP_df.sample(n=min(2, len(TP_df)))
FN = FN_df.sample(n=min(2, len(FN_df)))
FP = FP_df.sample(n=min(2, len(FP_df)))
TN = TN_df.sample(n=min(2, len(TN_df)))


# Function to format text into LaTeX paragraphs with correct chronological order
def format_latex_paragraphs(df):
    formatted_text = ""
    for _, row in df.iterrows():
        dialogue = row['text'].split("\n")

        # Ensure chronological order
        person1_dialogue = [line for line in dialogue if line.startswith("Person1")]
        person2_dialogue = [line for line in dialogue if line.startswith("Person2")]

        # Interleave dialogues between Person1 and Person2 in chronological order
        dialogue_ordered = []
        person1_idx, person2_idx = 0, 0
        while person1_idx < len(person1_dialogue) or person2_idx < len(person2_dialogue):
            if person1_idx < len(person1_dialogue):
                dialogue_ordered.append("\\noindent\\textbf{Person1:} " + person1_dialogue[person1_idx][8:] + " \\\\")
                person1_idx += 1
            if person2_idx < len(person2_dialogue):
                dialogue_ordered.append("\\noindent\\textbf{Person2:} " + person2_dialogue[person2_idx][8:] + " \\\\")
                person2_idx += 1

        formatted_text += " \n".join(dialogue_ordered) + "\n\n"
    return formatted_text

# Print examples as LaTeX paragraphs in chronological order

print("\n\\subsection{True Positives:}")
print("{\\ttfamily \\tiny")
print(format_latex_paragraphs(TP))
print("}")

print("\n\\subsection{False Negatives:}")
print("{\\ttfamily \\tiny")
print(format_latex_paragraphs(FN))
print("}")

print("\n\\subsection{False Positives:}")
print("{\\ttfamily \\tiny")
print(format_latex_paragraphs(FP))
print("}")

print("\n\\subsection{True Negatives:}")
print("{\\ttfamily \\tiny")
print(format_latex_paragraphs(TN))
print("}")

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



\subsection{True Positives:}
{\ttfamily \tiny
\noindent\textbf{Person1:}  Is he home right now? \\ 
\noindent\textbf{Person2:}  Yeah. \\ 
\noindent\textbf{Person1:}  Well, I think I can help a little more if I talk to him directly. \\ 
\noindent\textbf{Person2:}  I don't know -- \\ 
\noindent\textbf{Person1:}  I'm sure he won't be angry once he realizes how concerned you are about him. \\ 
\noindent\textbf{Person2:}  Okay, but if I get yelled at, I'm never gonna listen to this show again. \\ 
\noindent\textbf{Person1:}  Fair enough. \\

\noindent\textbf{Person1:}  I've noticed you've been coming home late recently. Is there something going on? \\ 
\noindent\textbf{Person2:}  Work has been really demanding. \\ 
\noindent\textbf{Person1:}  This is just like last year when you were hiding things from me. \\ 
\noindent\textbf{Person2:}  I’m not going to go through this again. I told you, it's just work. \\ 
\noindent\textbf{Person1:}  If we don't communicate, we'll end up in the same situ

In [22]:
# Load dataset

from datasets import load_dataset, ClassLabel
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
import numpy as np


# Load dataset from CSV on Google Drive
dataset = load_dataset(
    "csv",
    data_files={
        "test": "Ling.csv"
    }
)


# After dataset is loaded
original_labels = set(dataset["test"]["label"])
print("Original labels:", original_labels)  # Should print: {'manipulative', 'non-manipulative'}

# Define label2id and id2label
unique_labels = list(set(dataset["test"]["label"]))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode_labels)


def compute_metrics(pred):
    y_pred = np.argmax(pred.predictions, axis=1)
    y_true = pred.label_ids

    # Hardcoded label names since 0 = non-manipulative, 1 = manipulative
    target_names = ["non-manipulative", "manipulative"]

    print("✅ target_names:", target_names)
    print(classification_report(y_true, y_pred, target_names=target_names))
    return {}

# Initialize tokenizer
model_ckpt = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Tokenize using the 'body' column
def tokenize_fn(example):
    return tokenizer(example["body"], truncation=True, padding="max_length", max_length=128)


#def tokenize_fn(batch):
#    # Ensure all entries are strings
#    texts = [str(x) for x in batch["body"]]
#    return tokenizer(texts, truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


model_path = "./saved_roberta_lora_model"

from peft import PeftModel
base_model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)
model = PeftModel.from_pretrained(base_model, model_path)

## Training args

training_args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=16,
    do_train=False,
    do_eval=False,  # Turn off eval mode
    logging_dir="./logs",
    report_to="none",


)


# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics
)

# Predict and evaluate
preds = trainer.predict(tokenized["test"])

Original labels: {0, 1}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ target_names: ['non-manipulative', 'manipulative']
                  precision    recall  f1-score   support

non-manipulative       0.94      0.09      0.17      2401
    manipulative       0.17      0.97      0.29       458

        accuracy                           0.23      2859
       macro avg       0.56      0.53      0.23      2859
    weighted avg       0.82      0.23      0.19      2859



In [26]:
# Get predictions
predictions = trainer.predict(tokenized["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

# Decode input_ids from the tokenized test set
decoded_texts = tokenizer.batch_decode(tokenized["test"]["input_ids"], skip_special_tokens=True)

# Create DataFrame with predictions and text
results_df = pd.DataFrame({
    "text": decoded_texts,
    "true_label": y_true,
    "predicted_label": y_pred
})

# Identify true positives, false negatives, false positives, and true negatives
TP_df = results_df[(results_df.true_label == 1) & (results_df.predicted_label == 1)]
FN_df = results_df[(results_df.true_label == 1) & (results_df.predicted_label == 0)]
FP_df = results_df[(results_df.true_label == 0) & (results_df.predicted_label == 1)]
TN_df = results_df[(results_df.true_label == 0) & (results_df.predicted_label == 0)]

# Sample up to 2 examples from each category, if available
TP = TP_df.sample(n=min(2, len(TP_df)))
FN = FN_df.sample(n=min(2, len(FN_df)))
FP = FP_df.sample(n=min(2, len(FP_df)))
TN = TN_df.sample(n=min(2, len(TN_df)))

pd.set_option('display.max_colwidth', None)
print(TN)


import re


def parse_email_to_latex_dialogue(df):
    formatted_text = ""

    for _, row in df.iterrows():
        raw_text = row['text']
        lines = raw_text.strip().split("\\n")

        dialogue = []
        speaker = "Mail-1"

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Keep metadata with speaker tags
            if re.match(r"(URL|Date)", line):
                dialogue.append(f"{speaker}: [{line}]")
                continue

            # Alternate speakers on new chunks
            if dialogue and not line.startswith("[") and not line.startswith("http"):
                speaker = "Mail-2" if speaker == "Mail-1" else "Mail-1"

            dialogue.append(f"{speaker}: {line}")

        # Now format as LaTeX
        mail1_dialogue = [l for l in dialogue if l.startswith("Mail-1")]
        mail2_dialogue = [l for l in dialogue if l.startswith("Mail-2")]

        latex_lines = []
        idx1, idx2 = 0, 0
        while idx1 < len(mail1_dialogue) or idx2 < len(mail2_dialogue):
            if idx1 < len(mail1_dialogue):
                latex_lines.append("\\noindent\\textbf{Mail-1:} " + mail1_dialogue[idx1][8:] + " \\\\")
                idx1 += 1
            if idx2 < len(mail2_dialogue):
                latex_lines.append("\\noindent\\textbf{Mail-2:} " + mail2_dialogue[idx2][8:] + " \\\\")
                idx2 += 1

        formatted_text += "\n".join(latex_lines) + "\n\n"

    return formatted_text


print("\n\\subsection{True Positives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(TP))
print("}")

print("\n\\subsection{False Negatives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(FN))
print("}")

print("\n\\subsection{False Positives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(FP))
print("}")

print("\n\\subsection{True Negatives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(TN))
print("}")


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ target_names: ['non-manipulative', 'manipulative']
                  precision    recall  f1-score   support

non-manipulative       0.94      0.09      0.17      2401
    manipulative       0.17      0.97      0.29       458

        accuracy                           0.23      2859
       macro avg       0.56      0.53      0.23      2859
    weighted avg       0.82      0.23      0.19      2859

                                                                                                                                                                                                                                                                                                                                                                                                                                                                      text  \
2174                                                                                                                                  

In [27]:
  # Load dataset

from datasets import load_dataset, ClassLabel
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
import numpy as np


# Load dataset from CSV on Google Drive
dataset = load_dataset(
    "csv",
    data_files={
        "test": "SpamAssasin.csv"
    }
)


# After dataset is loaded
original_labels = set(dataset["test"]["label"])
print("Original labels:", original_labels)  # Should print: {'manipulative', 'non-manipulative'}

# Define label2id and id2label based on unique labels in the dataset
unique_labels = list(set(dataset["test"]["label"]))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}


def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode_labels)


def compute_metrics(pred):
    y_pred = np.argmax(pred.predictions, axis=1)
    y_true = pred.label_ids

    # Hardcoded label names since 0 = non-manipulative, 1 = manipulative
    target_names = ["non-manipulative", "manipulative"]

    print("✅ target_names:", target_names)
    print(classification_report(y_true, y_pred, target_names=target_names))
    return {}

# Initialize tokenizer
model_ckpt = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(batch):
    # Ensure all entries are strings
    texts = [str(x) for x in batch["body"]]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Load model
from peft import PeftModel
base_model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)
model = PeftModel.from_pretrained(base_model, model_path)
## Training args

training_args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=16,
    do_train=False,
    do_eval=False,  # Turn off eval mode
    logging_dir="./logs",
    report_to="none",

)


# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics
)

# Predict and evaluate
preds = trainer.predict(tokenized["test"])

Original labels: {0, 1}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ target_names: ['non-manipulative', 'manipulative']
                  precision    recall  f1-score   support

non-manipulative       0.91      0.22      0.36      4091
    manipulative       0.34      0.94      0.50      1718

        accuracy                           0.44      5809
       macro avg       0.62      0.58      0.43      5809
    weighted avg       0.74      0.44      0.40      5809



In [28]:
# Get predictions
predictions = trainer.predict(tokenized["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

# Decode input_ids from the tokenized test set
decoded_texts = tokenizer.batch_decode(tokenized["test"]["input_ids"], skip_special_tokens=True)

# Create DataFrame with predictions and text
results_df = pd.DataFrame({
    "text": decoded_texts,
    "true_label": y_true,
    "predicted_label": y_pred
})

# Identify true positives, false negatives, false positives, and true negatives
TP_df = results_df[(results_df.true_label == 1) & (results_df.predicted_label == 1)]
FN_df = results_df[(results_df.true_label == 1) & (results_df.predicted_label == 0)]
FP_df = results_df[(results_df.true_label == 0) & (results_df.predicted_label == 1)]
TN_df = results_df[(results_df.true_label == 0) & (results_df.predicted_label == 0)]

# Sample up to 2 examples from each category, if available
TP = TP_df.sample(n=min(2, len(TP_df)))
FN = FN_df.sample(n=min(2, len(FN_df)))
FP = FP_df.sample(n=min(2, len(FP_df)))
TN = TN_df.sample(n=min(2, len(TN_df)))

pd.set_option('display.max_colwidth', None)
print(TN)


import re


def parse_email_to_latex_dialogue(df):
    formatted_text = ""

    for _, row in df.iterrows():
        raw_text = row['text']
        lines = raw_text.strip().split("\\n")

        dialogue = []
        speaker = "Mail-1"

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Keep metadata with speaker tags
            if re.match(r"(URL|Date)", line):
                dialogue.append(f"{speaker}: [{line}]")
                continue

            # Alternate speakers on new chunks
            if dialogue and not line.startswith("[") and not line.startswith("http"):
                speaker = "Mail-2" if speaker == "Mail-1" else "Mail-1"

            dialogue.append(f"{speaker}: {line}")

        # Now format as LaTeX
        mail1_dialogue = [l for l in dialogue if l.startswith("Mail-1")]
        mail2_dialogue = [l for l in dialogue if l.startswith("Mail-2")]

        latex_lines = []
        idx1, idx2 = 0, 0
        while idx1 < len(mail1_dialogue) or idx2 < len(mail2_dialogue):
            if idx1 < len(mail1_dialogue):
                latex_lines.append("\\noindent\\textbf{Mail-1:} " + mail1_dialogue[idx1][8:] + " \\\\")
                idx1 += 1
            if idx2 < len(mail2_dialogue):
                latex_lines.append("\\noindent\\textbf{Mail-2:} " + mail2_dialogue[idx2][8:] + " \\\\")
                idx2 += 1

        formatted_text += "\n".join(latex_lines) + "\n\n"

    return formatted_text


print("\n\\subsection{True Positives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(TP))
print("}")

print("\n\\subsection{False Negatives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(FN))
print("}")

print("\n\\subsection{False Positives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(FP))
print("}")

print("\n\\subsection{True Negatives:}")
print("{\\ttfamily \\tiny")
print(parse_email_to_latex_dialogue(TN))
print("}")


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ target_names: ['non-manipulative', 'manipulative']
                  precision    recall  f1-score   support

non-manipulative       0.91      0.22      0.36      4091
    manipulative       0.34      0.94      0.50      1718

        accuracy                           0.44      5809
       macro avg       0.62      0.58      0.43      5809
    weighted avg       0.74      0.44      0.40      5809

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      text  \
3815                                                                                  